# CNN for MNIST Handwritten Digit Recognition

In [1]:
#importing nessasary libraies
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

### Downloading MNIST

In [2]:
#Downloading MNIST
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root = "./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root = "./data",
    train=False,
    download=True,
    transform=transform
)    

100.0%
100.0%
100.0%
100.0%


### Create Dataloaders

In [4]:
#Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

### Check Image Shape

In [5]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([64, 1, 28, 28])
torch.Size([64])


### Building first CNN

In [10]:
class CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels = 1,
            out_channels = 16,
            kernel_size = 3,
            padding = 1
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.conv2 = nn.Conv2d(
            in_channels = 16,
            out_channels = 32,
            kernel_size = 3,
            padding = 1
        )

        self.fc1 = nn.Linear(
            32 * 7 * 7,
            10
        )

    def forward(self, x):

            x = self.pool(torch.relu(self.conv1(x)))

            x = self.pool(torch.relu(self.conv2(x)))

            x = x.view(x.size(0), -1)

            x = self.fc1(x)
            
        
            return x

### Create Model

In [11]:
model = CNN()

print(model)

CNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=1568, out_features=10, bias=True)
)


### Verify output Shape

In [12]:
sample_images, _ = next(iter(train_loader))

output = model(sample_images)

print(output.shape)

torch.Size([64, 10])


### Loss and Optimizer

In [18]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr = 0.001
)

### Training Loop

In [22]:
epochs = 5

for epoch in range(epochs):

    running_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1/5, Loss: 0.0036
Epoch 2/5, Loss: 0.0024
Epoch 3/5, Loss: 0.0019
Epoch 4/5, Loss: 0.0004
Epoch 5/5, Loss: 0.0038


### Evaluate Accuracy

In [23]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 98.92%


### Saving the Model

In [24]:
torch.save(model.state_dict(), "mnist_cnn.pth")

print("Model saved successfully!")

Model saved successfully!


In [26]:
import os
print(os.getcwd())

C:\Users\Maanvi\Projects\IrisFlowerProject
